In [1]:
import os
print(os.listdir('/kaggle/input'))

['models', 'competitions']


In [2]:
# ============================================================
# INFERENCE NOTEBOOK — Config
# Edit INFER_THRESH, RADIUS_XY, RADIUS_Z here and re-run Cell 1
# then jump to Cell 5 (no need to reload model) to re-generate submission
# ============================================================
import os, json, glob
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import cv2
from scipy.ndimage import maximum_filter
from scipy.optimize import linear_sum_assignment
from skimage.feature import peak_local_max

# ---- device ----
DEVICE = 'cpu'
if torch.cuda.is_available():
    try:
        _p = nn.Conv2d(1, 1, 3).cuda()
        _ = _p(torch.zeros(1, 1, 8, 8, device='cuda')).cpu()
        del _p
        DEVICE = 'cuda'
    except Exception as e:
        print(f'GPU unusable: {str(e)[:80]} -> CPU')
print(f'Device: {DEVICE}')

# ---- paths ----
COMP_ROOT = '/kaggle/input/competitions/biohub-cell-tracking-during-development'
TEST_DIR  = Path(COMP_ROOT) / 'test'
OUT_DIR   = Path('/kaggle/working')
MODELS_DIR = Path('/kaggle/input/models')

# ---- load metadata from unet2d_meta.json if it exists ----
meta_path = next(MODELS_DIR.rglob('*.json'), None)
if meta_path is not None:
    with open(meta_path) as f:
        saved_meta = json.load(f)
    print(f'Meta loaded from {meta_path}:')
    print(saved_meta)
    BASE         = int(saved_meta.get('base',         32))
    POOL         = int(saved_meta.get('pool',          4))
    INFER_THRESH = float(saved_meta.get('best_thresh',
               saved_meta.get('detect_thresh', 0.35)))
    VAL_RECALL   = saved_meta.get('val_recall', 'unknown')
else:
    # fallback: infer BASE from weight tensor shape
    print('No meta JSON found -- inferring BASE from checkpoint weights')
    _raw = torch.load(next(MODELS_DIR.rglob('*.pt')), map_location='cpu')
    BASE = int(_raw['e1.0.weight'].shape[0])
    POOL = 4          # confirmed from training config
    INFER_THRESH = 0.35
    VAL_RECALL   = 'unknown'
    print(f'Inferred BASE={BASE} from e1.0.weight shape {tuple(_raw["e1.0.weight"].shape)}')

print(f'BASE={BASE} | POOL={POOL} | INFER_THRESH={INFER_THRESH} | val_recall={VAL_RECALL}')

# ---- load checkpoint ----
CKPT_PATH = next(MODELS_DIR.rglob('unet2d_best.pt'),
            next(MODELS_DIR.rglob('*.pt'), None))
if CKPT_PATH is None:
    raise FileNotFoundError(f'No .pt file found under {MODELS_DIR}')
print(f'Loading: {CKPT_PATH}')

# raw state dict -- load directly into model after model is defined in Cell 2
raw_state_dict = torch.load(CKPT_PATH, map_location='cpu')
print(f'State dict keys: {len(raw_state_dict)} tensors')

SCALE = np.array([1.625, 0.40625, 0.40625])
print(f'SCALE={SCALE} | TEST_DIR exists={TEST_DIR.exists()}')

Device: cuda
Meta loaded from /kaggle/input/models/sumantvj/biohub-cell-detection/pytorch/default/7/unet2d_meta.json:
{'base': 16, 'pool': 2, 'n_context': 3, 'gauss_sigma': 4.0, 'best_thresh': 0.05, 'best_val_recall': 0.5990990990990991, 'val_recall': 0.5990990990990991, 'min_peak_distance': 9, 'nucleus_diam_um': 8.0, 'clahe_clip': 3.0, 'clahe_grid': 8, 'w_pos': 15.0, 'w_bg': 1.0, 'w_ign': 0.01, 'z_margin': 5, 'neg_slices_per_t': 4, 'epochs_trained': 40, 'nms_radius_xy': 6.0, 'nms_radius_z': 3, 'detect_thresh': 0.05}
BASE=16 | POOL=2 | INFER_THRESH=0.05 | val_recall=0.5990990990990991
Loading: /kaggle/input/models/sumantvj/biohub-cell-detection/pytorch/default/7/unet2d_best.pt
State dict keys: 76 tensors
SCALE=[1.625   0.40625 0.40625] | TEST_DIR exists=True


In [3]:
# Must match the architecture used in training exactly
# If you changed BASE in the training notebook, update it here too
def _block2d(ci, co):
    return nn.Sequential(
        nn.Conv2d(ci, co, 3, padding=1), nn.BatchNorm2d(co), nn.ReLU(inplace=True),
        nn.Conv2d(co, co, 3, padding=1), nn.BatchNorm2d(co), nn.ReLU(inplace=True),
    )

class UNet2p5D(nn.Module):
    def __init__(self, base=32, in_channels=3):
        super().__init__()
        b = base
        self.e1   = _block2d(in_channels, b)
        self.e2   = _block2d(b,   b*2)
        self.pool = nn.MaxPool2d(2)
        self.bn   = _block2d(b*2, b*4)
        self.u2   = nn.ConvTranspose2d(b*4, b*2, 2, stride=2)
        self.d2   = _block2d(b*4, b*2)
        self.u1   = nn.ConvTranspose2d(b*2, b,   2, stride=2)
        self.d1   = _block2d(b*2, b)
        self.out  = nn.Conv2d(b, 1, 1)

    def forward(self, x):
        e1 = self.e1(x)
        e2 = self.e2(self.pool(e1))
        b  = self.bn(self.pool(e2))
        d2 = self.d2(torch.cat([self.u2(b),  e2], 1))
        d1 = self.d1(torch.cat([self.u1(d2), e1], 1))
        return self.out(d1)

model = UNet2p5D(base=BASE, in_channels=3).to(DEVICE)
model.load_state_dict(raw_state_dict)
model.eval()
print(f'UNet2p5D loaded: {sum(p.numel() for p in model.parameters()):,} params')

UNet2p5D loaded: 117,681 params


In [4]:
_ZC = {}

def read_meta(zp):
    with open(Path(zp) / '0' / 'zarr.json') as f:
        m = json.load(f)
    return dict(shape=tuple(m['shape']), dtype=np.dtype(m['data_type']))

def load_vol(zp, t, meta=None):
    try:
        import zarr
        k = str(zp)
        if k not in _ZC:
            _ZC[k] = zarr.open(k, mode='r')['0']
        return np.asarray(_ZC[k][t])
    except Exception:
        import blosc2
        if meta is None:
            meta = read_meta(zp)
        buf = blosc2.decompress(
            open(Path(zp) / '0' / 'c' / str(t) / '0' / '0' / '0', 'rb').read()
        )
        return np.frombuffer(buf, dtype=meta['dtype']).reshape(meta['shape'][1:])

def pool_xy(vol, f=POOL):
    Z, Y, X = vol.shape
    Y2, X2 = (Y // f) * f, (X // f) * f
    v = vol[:, :Y2, :X2].astype(np.float32, copy=False)
    return v.reshape(Z, Y2 // f, f, X2 // f, f).mean(axis=(2, 4))

def normalize_slice_clahe(slc, clip=2.0, grid=8):
    lo = float(np.percentile(slc, 2.0))
    hi = float(np.percentile(slc, 99.0))
    if hi <= lo:
        return np.zeros_like(slc, dtype=np.float32)
    scaled = np.clip((slc - lo) / (hi - lo) * 255.0, 0, 255).astype(np.uint8)
    clahe  = cv2.createCLAHE(clipLimit=clip, tileGridSize=(grid, grid))
    return clahe.apply(scaled).astype(np.float32) / 255.0

print('I/O helpers ready.')

I/O helpers ready.


In [5]:
# ---- inference config ----
NUCLEUS_DIAM_UM   = 8.0
VOXEL_XY_POOLED   = 0.40625 * POOL
MIN_PEAK_DISTANCE = max(5, int(NUCLEUS_DIAM_UM / VOXEL_XY_POOLED))
MIN_CONSECUTIVE_Z = 3
NMS_RADIUS_XY     = 6.0
NMS_RADIUS_Z      = 3

# threshold lowered from 0.35 -> 0.25
# training notebook Cell 12 shows:
#   thresh=0.40: recall=36.5%, avg_dets=150
#   thresh=0.42: recall=25.6% -- drops sharply above 0.40
# zero detections on 44b6_0b24845f means that dataset's heatmap
# max never reaches 0.35, so the fixed threshold kills it entirely.
# 0.25 gives a safe floor; adaptive_threshold raises it per-volume
# if the model is strongly activating, to avoid over-detection.
INFER_THRESH = 0.25

print(f'INFER_THRESH={INFER_THRESH} | MIN_PEAK_DISTANCE={MIN_PEAK_DISTANCE}')
print(f'MIN_CONSECUTIVE_Z={MIN_CONSECUTIVE_Z}')
print(f'NMS_RADIUS_XY={NMS_RADIUS_XY} | NMS_RADIUS_Z={NMS_RADIUS_Z}')

def get_2p5d_stack_clahe(pvol, z, clip=3.0, grid=8):
    """3-channel stack: [z-1, z, z+1] with CLAHE normalization."""
    Z = pvol.shape[0]
    slices = []
    for dz in [-1, 0, 1]:
        z_idx = int(np.clip(z + dz, 0, Z - 1))
        slc   = pvol[z_idx]
        lo    = float(np.percentile(slc, 2.0))
        hi    = float(np.percentile(slc, 99.0))
        if hi <= lo:
            slices.append(np.zeros_like(slc, dtype=np.float32))
            continue
        scaled = np.clip((slc - lo) / (hi - lo) * 255.0, 0, 255).astype(np.uint8)
        clahe  = cv2.createCLAHE(clipLimit=clip, tileGridSize=(grid, grid))
        slices.append(clahe.apply(scaled).astype(np.float32) / 255.0)
    return np.stack(slices, axis=0)  # (3, H, W)

def adaptive_threshold(pvol, base_thresh=INFER_THRESH):
    """
    Samples 8 slices, checks the model's max heatmap output.
    If the volume is dimmer than usual (max < base_thresh * 1.5),
    lowers the threshold proportionally so dim datasets still get detections.
    Never goes below 0.05 to avoid pure noise detections.
    """
    sample_zs = list(range(0, pvol.shape[0], max(1, pvol.shape[0]//8)))
    peak_vals = []
    model.eval()
    with torch.no_grad():
        for z in sample_zs:
            stack = get_2p5d_stack_clahe(pvol, z)
            xt    = torch.from_numpy(stack[None]).to(DEVICE)
            hm    = torch.sigmoid(model(xt))[0, 0].cpu().numpy()
            peak_vals.append(float(hm.max()))
    vol_max = float(np.median(peak_vals))
    if vol_max < base_thresh * 1.5:
        adaptive = max(0.05, vol_max * 0.5)
        print(f'    [adaptive] vol_max_median={vol_max:.3f} '
              f'-> threshold={adaptive:.3f} (base was {base_thresh:.3f})')
        return adaptive
    return base_thresh

def detect_volume_with_scores_2p5d(pvol, model, thresh=None,
                                    min_dist=MIN_PEAK_DISTANCE,
                                    min_consec_z=MIN_CONSECUTIVE_Z):
    if thresh is None:
        thresh = adaptive_threshold(pvol)

    model.eval()
    Z = pvol.shape[0]
    slice_dets   = {}
    slice_scores = {}

    with torch.no_grad():
        for z in range(Z):
            stack = get_2p5d_stack_clahe(pvol, z)        # (3, H, W)
            xt    = torch.from_numpy(stack[None]).to(DEVICE)  # (1, 3, H, W)
            hm    = torch.sigmoid(model(xt))[0, 0].cpu().numpy()

            if z == 0 or z % 20 == 0:
                print(f'      z={z:3d} hm=[{hm.min():.3f},{hm.max():.3f}] '
                      f'thresh={thresh:.3f}')

            pk = peak_local_max(hm, min_distance=min_dist,
                                 threshold_abs=thresh, exclude_border=False)
            slice_dets[z]   = pk
            slice_scores[z] = np.array([hm[p[0], p[1]] for p in pk]) \
                               if len(pk) > 0 else np.array([])

    confirmed = []
    scores    = []
    for z in range(Z):
        pk_z  = slice_dets.get(z,  np.zeros((0, 2)))
        scr_z = slice_scores.get(z, np.array([]))
        if len(pk_z) == 0:
            continue
        for idx, det_yx in enumerate(pk_z):
            consec = 1
            cur_yx = det_yx.copy()
            for dz in range(1, min_consec_z):
                pk_next = slice_dets.get(z + dz, np.zeros((0, 2)))
                if len(pk_next) == 0:
                    break
                dists = np.linalg.norm(pk_next - cur_yx, axis=1)
                ci    = np.argmin(dists)
                if dists[ci] <= NMS_RADIUS_XY * 2:
                    consec += 1
                    cur_yx  = pk_next[ci]
                else:
                    break
            if consec >= min_consec_z:
                confirmed.append((z, float(det_yx[0]), float(det_yx[1])))
                scores.append(float(scr_z[idx]) if idx < len(scr_z) else 0.0)

    return confirmed, scores

def nms_3d_weighted(confirmed_dets, scores,
                     radius_xy=NMS_RADIUS_XY, radius_z=NMS_RADIUS_Z):
    """
    3D NMS with score-weighted centroid -- replaces nms_3d entirely.
    Same inputs/outputs as nms_3d but returns (centroids, scores) tuple.
    """
    if len(confirmed_dets) == 0:
        return [], []
    dets  = np.array(confirmed_dets, dtype=np.float64)
    scrs  = np.array(scores,         dtype=np.float64)
    order = np.argsort(scrs)[::-1]
    keep_centroids = []
    keep_scores    = []
    suppressed     = np.zeros(len(dets), dtype=bool)
    for i in order:
        if suppressed[i]:
            continue
        zi, yi, xi = dets[i]
        cluster_idx = []
        for j in range(len(dets)):
            if suppressed[j]:
                continue
            zj, yj, xj = dets[j]
            if abs(zi-zj) <= radius_z and \
               np.sqrt((yi-yj)**2 + (xi-xj)**2) <= radius_xy:
                cluster_idx.append(j)
                suppressed[j] = True
        c_dets = dets[cluster_idx]; c_scrs = scrs[cluster_idx]
        tw = c_scrs.sum()
        if tw > 0:
            z_wt = float(np.dot(c_scrs, c_dets[:,0]) / tw)
            y_wt = float(np.dot(c_scrs, c_dets[:,1]) / tw)
            x_wt = float(np.dot(c_scrs, c_dets[:,2]) / tw)
        else:
            z_wt, y_wt, x_wt = float(zi), float(yi), float(xi)
        keep_centroids.append((z_wt, y_wt, x_wt))
        keep_scores.append(float(c_scrs.max()))
    return keep_centroids, keep_scores

print('Detection functions ready.')
print(f'Using 2.5D stacking (3-channel input) matching training architecture.')

INFER_THRESH=0.25 | MIN_PEAK_DISTANCE=9
MIN_CONSECUTIVE_Z=3
NMS_RADIUS_XY=6.0 | NMS_RADIUS_Z=3
Detection functions ready.
Using 2.5D stacking (3-channel input) matching training architecture.


In [6]:
def link_timepoints_advanced(nodes_t0, nodes_t1):
    if not nodes_t0 or not nodes_t1:
        return []
        
    # Extract physical locations
    pos0_um = np.array([[n['phys_z'], n['phys_y'], n['phys_x']] for n in nodes_t0])
    pos1_um = np.array([[n['phys_z'], n['phys_y'], n['phys_x']] for n in nodes_t1])

    # 1. Motion prediction based on past trajectory
    predicted_pos1_um = []
    for i, n in enumerate(nodes_t0):
        if n.get('parent_pos_um') is not None:
            velocity = pos0_um[i] - n['parent_pos_um']
            velocity = np.clip(velocity, -4.0, 4.0) # Prevent extreme jump predictions
            predicted_pos1_um.append(pos0_um[i] + velocity)
        else:
            predicted_pos1_um.append(pos0_um[i])
    predicted_pos1_um = np.array(predicted_pos1_um)

    # 2. Cost Matrix with Confidence Penalties
    dist_matrix = np.linalg.norm(predicted_pos1_um[:, None, :] - pos1_um[None, :, :], axis=2)
    scores_t1 = np.array([n['score'] for n in nodes_t1])
    confidence_penalty = 2.0 * (1.0 - scores_t1)[None, :]
    cost_matrix = dist_matrix + confidence_penalty

    # Hard cutoff distance
    MAX_DIST = 15.0 
    cost_matrix[dist_matrix > MAX_DIST] = 1e5
    row_ind, col_ind = linear_sum_assignment(cost_matrix)

    links = []
    matched_curr = set()
    prev_to_curr = {}

    for r, c in zip(row_ind, col_ind):
        if dist_matrix[r, c] <= MAX_DIST:
            links.append({
                'source_id': nodes_t0[r]['id'],
                'target_id': nodes_t1[c]['id'],
                'source_pos_um': pos0_um[r]
            })
            matched_curr.add(c)
            prev_to_curr.setdefault(r, []).append(c)
            
    # 3. Stage 2: Division check (1 parent -> 2 daughters)
    DIV_MAX_DIST = 8.0
    for ci, n1 in enumerate(nodes_t1):
        if ci in matched_curr: continue
            
        best_r, best_d = None, DIV_MAX_DIST
        for ri, n0 in enumerate(nodes_t0):
            if ri not in prev_to_curr or len(prev_to_curr[ri]) != 1: continue
                
            d = np.linalg.norm(pos0_um[ri] - pos1_um[ci])
            if d < best_d:
                existing_ci = prev_to_curr[ri][0]
                daughter_dist = np.linalg.norm(pos1_um[ci] - pos1_um[existing_ci])
                if daughter_dist >= 3.0: # Minimum biological separation for a real split
                    best_d = d
                    best_r = ri
                    
        if best_r is not None:
            links.append({
                'source_id': nodes_t0[best_r]['id'],
                'target_id': nodes_t1[ci]['id'],
                'source_pos_um': pos0_um[best_r]
            })
            
    return links

In [7]:
import networkx as nx

test_names = sorted([d.replace('.zarr', '') for d in os.listdir(TEST_DIR) if d.endswith('.zarr')])
print(f'Test datasets: {len(test_names)}')
all_rows = []

for folder_name in test_names:
    zp = str(TEST_DIR / f'{folder_name}.zarr')
    meta = read_meta(zp)
    n_t = meta['shape'][0]
    print(f'\n{folder_name} | {n_t} timepoints')

    all_nodes_by_t = {}
    edges_list = []
    global_node_counter = 0

    # Pass 1: Sequential Detection and Tracking
    for t in range(n_t):
        vol = load_vol(zp, t, meta)
        pvol = pool_xy(vol)

        confirmed, scores = detect_volume_with_scores_2p5d(pvol, model)
        nuclei, nms_scores = nms_3d_weighted(confirmed, scores)

        nodes_t = []
        for i, (z, py, px) in enumerate(nuclei):
            orig_y, orig_x = py * POOL, px * POOL
            phys_z, phys_y, phys_x = z * SCALE[0], orig_y * SCALE[1], orig_x * SCALE[2]
            
            nodes_t.append({
                'id': global_node_counter,
                't': t, 'z': z, 'y': orig_y, 'x': orig_x,
                'phys_z': phys_z, 'phys_y': phys_y, 'phys_x': phys_x,
                'score': nms_scores[i],
                'parent_pos_um': None # Will be populated if linked
            })
            global_node_counter += 1
            
        all_nodes_by_t[t] = nodes_t
        if t % 20 == 0 or t == n_t - 1:
            print(f'  t={t:3d}/{n_t-1} | {len(nuclei)} nuclei detected')

        # Link dynamically based on physical scale and past trajectory
        if t > 0:
            links = link_timepoints_advanced(all_nodes_by_t[t-1], all_nodes_by_t[t])
            for link in links:
                edges_list.append((link['source_id'], link['target_id']))
                # Propagate position to target for next frame's velocity calculation
                for n in all_nodes_by_t[t]:
                    if n['id'] == link['target_id']:
                        n['parent_pos_um'] = link['source_pos_um']
                        break

    # Pass 2: Filter noise using NetworkX
    G = nx.Graph()
    for t, nodes in all_nodes_by_t.items():
        for n in nodes:
            G.add_node(n['id'], data=n)
    for src, tgt in edges_list:
        G.add_edge(src, tgt)

    keep_set = set()
    for comp in nx.connected_components(G):
        if len(comp) >= 3: # Keep valid tracks spanning >= 3 frames
            keep_set.update(comp)

    # Pass 3: Export to CSV layout
    for t, nodes in all_nodes_by_t.items():
        for n in nodes:
            if n['id'] in keep_set:
                all_rows.append({
                    'dataset': folder_name, 'row_type': 'node',
                    'node_id': n['id'], 't': n['t'], 'z': int(round(n['z'])), 
                    'y': int(round(n['y'])), 'x': int(round(n['x'])),
                    'source_id': -1, 'target_id': -1,
                })

    for src, tgt in edges_list:
        if src in keep_set and tgt in keep_set:
            all_rows.append({
                'dataset': folder_name, 'row_type': 'edge',
                'node_id': -1, 't': -1, 'z': -1, 'y': -1, 'x': -1,
                'source_id': src, 'target_id': tgt,
            })
            
    n_nodes = sum(1 for r in all_rows if r['dataset'] == folder_name and r['row_type'] == 'node')
    print(f'  -> kept: {n_nodes} nodes')

# Compile and Save
submission = pd.DataFrame(all_rows, columns=['dataset', 'row_type', 'node_id', 't', 'z', 'y', 'x', 'source_id', 'target_id'])
submission.index.name = 'id'
sub_path = OUT_DIR / 'submission.csv'
submission.to_csv(sub_path)

print(f'\n{"="*50}\nSubmission Ready: {sub_path}')
print(f'Total Tracker Rows: {len(submission)}')

Test datasets: 4

44b6_0113de3b | 100 timepoints
      z=  0 hm=[0.005,0.661] thresh=0.250
      z= 20 hm=[0.004,0.716] thresh=0.250
      z= 40 hm=[0.005,0.703] thresh=0.250
      z= 60 hm=[0.005,0.732] thresh=0.250
  t=  0/99 | 80 nuclei detected
      z=  0 hm=[0.005,0.730] thresh=0.250
      z= 20 hm=[0.005,0.657] thresh=0.250
      z= 40 hm=[0.005,0.727] thresh=0.250
      z= 60 hm=[0.005,0.695] thresh=0.250
      z=  0 hm=[0.005,0.729] thresh=0.250
      z= 20 hm=[0.004,0.683] thresh=0.250
      z= 40 hm=[0.006,0.739] thresh=0.250
      z= 60 hm=[0.005,0.779] thresh=0.250
      z=  0 hm=[0.004,0.710] thresh=0.250
      z= 20 hm=[0.004,0.755] thresh=0.250
      z= 40 hm=[0.005,0.683] thresh=0.250
      z= 60 hm=[0.005,0.488] thresh=0.250
      z=  0 hm=[0.006,0.696] thresh=0.250
      z= 20 hm=[0.004,0.623] thresh=0.250
      z= 40 hm=[0.005,0.662] thresh=0.250
      z= 60 hm=[0.006,0.784] thresh=0.250
      z=  0 hm=[0.005,0.724] thresh=0.250
      z= 20 hm=[0.004,0.683] thresh=0